# Export MODIS IGBP Land Cover — Preprocessing Step 4

**Version:** 1.0 (2026-03-10) © R. Butzer

## What this notebook does
1. **(Phase 1 — GEE)** Exports MODIS MCD12Q1 IGBP (2001–2024) as 24-band GeoTIFF via Google Drive
2. **(Phase 2 — Local)** PyDrive download + validation against grid reference

## Inputs
- GEE Collection: `MODIS/061/MCD12Q1`
- GEE Asset: `projects/ee-butzerfe/assets/wilde_only_grid_glance_eu_150km`
- `_Runs\02_Woody_Cover\woody_cover_500m_3035.tif` — grid reference for validation

## Outputs
- `_Runs\05_Landcover_integrated\MODIS_IGBP_2001_2024_500m_wildE.tif` — 24 bands (2001–2024)
  - → **Input for `05_analysis_eco_lc.ipynb`**

> **Note:** This notebook replaces the former „Phase 2 Kombination" step from
> `05_landcover_analysis.ipynb` which combined Woody, MBA and MODIS into a single
> `woody_burned_MBA_MODIS_IGBP_combined.tif`. That combined TIF is no longer used —
> each analysis step reads the three source files directly.


In [2]:
import ee
import datetime
import os
from tqdm import tqdm
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
import rasterio

# Earth Engine initialisieren
try:
    ee.Initialize()
except Exception:
    ee.Authenticate()
    ee.Initialize()

workDir = r"\\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis"

print("✓ Imports abgeschlossen")
print("✓ Earth Engine initialisiert")
print(f"✓ workDir: {workDir}")



Successfully saved authorization token.
✓ Imports abgeschlossen
✓ Earth Engine initialisiert
✓ workDir: \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis


In [3]:
# === MODIS Land Cover laden ===

dataset = ee.ImageCollection('MODIS/061/MCD12Q1')

def get_modis_lc_period(year):
    """Lade MODIS IGBP Land Cover für ein bestimmtes Jahr"""
    yearly_img = dataset.filter(
        ee.Filter.calendarRange(year, year, 'year')
    ).first().select('LC_Type1').rename(f'MODIS_LC_{year}')
    return yearly_img

# Lade MODIS für ALLE Jahre von 2001-2024
years = list(range(2001, 2025))  # [2001, 2002, ..., 2024]

modis_lc_images = []
for year in years:
    try:
        img = get_modis_lc_period(year)
        modis_lc_images.append(img)
    except Exception as e:
        print(f"⚠️  Fehler beim Laden von {year}: {str(e)}")

print(f"✓ {len(modis_lc_images)} MODIS IGBP Jahre geladen (2001-2024)")

✓ 24 MODIS IGBP Jahre geladen (2001-2024)


In [4]:
# === Clippe auf wildE-Grid & Reproject zu 500m ===

grid_ee = ee.FeatureCollection("projects/ee-butzerfe/assets/wilde_only_grid_glance_eu_150km")
region = grid_ee.geometry()

# Kombiniere alle Jahre und reproject zu 500m
modis_lc_clipped = ee.Image.cat([
    img.reproject(
        crs='EPSG:3035',
        scale=500
    ).clip(region)
    for img in modis_lc_images
])

print("✓ Auf wildE-Grid geclippt und zu 500m reprojektion")
print(f"✓ Gesamte Bänder im Export: {len(years)}")

✓ Auf wildE-Grid geclippt und zu 500m reprojektion
✓ Gesamte Bänder im Export: 24


In [5]:
# === Export zu Google Drive ===

task = ee.batch.Export.image.toDrive(
    image=modis_lc_clipped,
    description='MODIS_IGBP_2001_2024_500m_wildE',
    folder=f'MODIS_GEE_{datetime.datetime.now().strftime("%Y%m%d_%H%M%S")}',
    fileNamePrefix='MODIS_IGBP_2001_2024_500m_wildE',
    region=region,
    crs='EPSG:3035',
    scale=500,
    maxPixels=1e13,
    fileFormat='GeoTIFF'
)

task.start()
print("✓✓✓ MODIS IGBP 24-Jahres-Export gestartet!")
print(f"Task ID: {task.id}")
print(f"Jahre im Export: 2001-2024 (24 Bänder)")
print("→ Export-Status unter: https://code.earthengine.google.com/tasks")

✓✓✓ MODIS IGBP 24-Jahres-Export gestartet!
Task ID: 2KAQ7YNI5WJCLPYPQ2XNZ2IM
Jahre im Export: 2001-2024 (24 Bänder)
→ Export-Status unter: https://code.earthengine.google.com/tasks


## Phase 2 — Lokaler Download & Validierung

In [6]:
# === Download MODIS IGBP Export von Google Drive ===

print("Starte Google Drive Authentifizierung...")

client_secrets_path = r"\\141.20.141.12\SAN_BioGeo\_HiWi\Ruben\wildE\client_secrets_ruben_geopy.json"

if not os.path.exists(client_secrets_path):
    print(f"⚠️  Warnung: Secrets nicht gefunden unter {client_secrets_path}")
    gauth = GoogleAuth()
else:
    print(f"✓ Secrets gefunden: {client_secrets_path}")
    GoogleAuth.DEFAULT_SETTINGS['client_config_file'] = client_secrets_path
    gauth = GoogleAuth()

try:
    gauth.LocalWebserverAuth()
    drive = GoogleDrive(gauth)
    print("✓ Google Drive Authentifizierung erfolgreich!")
except Exception as e:
    print(f"✗ Authentifizierungsfehler: {str(e)}")
    raise

# Zielordner
dl_folder = os.path.join(workDir, r"_Runs\04_Landcover\MODIS_GEE_tiles")
os.makedirs(dl_folder, exist_ok=True)
print(f"✓ Download-Ordner: {dl_folder}")


Starte Google Drive Authentifizierung...
✓ Secrets gefunden: \\141.20.141.12\SAN_BioGeo\_HiWi\Ruben\wildE\client_secrets_ruben_geopy.json
Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?client_id=579762042005-bktd78digk65s9sht4hho28rq5s30apo.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8080%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive&access_type=offline&response_type=code

Authentication successful.
✓ Google Drive Authentifizierung erfolgreich!
✓ Download-Ordner: \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\04_Landcover\MODIS_GEE_tiles


In [7]:
# === Suche und Download MODIS_GEE_*-Ordner ===

print("Suche nach MODIS_GEE_* Export-Ordnern auf Google Drive...")

folder_query = "mimeType='application/vnd.google-apps.folder' and trashed=false"
all_folders = drive.ListFile({'q': folder_query}).GetList()
folder_list = [f for f in all_folders if 'MODIS_GEE_' in f['title']]

if len(folder_list) == 0:
    print("✗ Keine MODIS_GEE_* Ordner gefunden!")
    print("Überprüfe, ob der GEE-Export abgeschlossen ist.")
else:
    print(f"✓ {len(folder_list)} MODIS_GEE_* Ordner gefunden:")
    for i, f in enumerate(folder_list, 1):
        print(f"  {i}. {f['title']} (ID: {f['id']})")

    folder_list_sorted = sorted(folder_list, key=lambda x: x['title'], reverse=True)
    selected_folder = folder_list_sorted[0]
    print(f"\n✓ Neuester Ordner: {selected_folder['title']}")

    gee_folder_id = selected_folder['id']
    export_folder_name = selected_folder['title']

    drive_list = drive.ListFile({'q': f"'{gee_folder_id}' in parents and trashed=false"}).GetList()
    print(f"Anzahl Dateien im Ordner: {len(drive_list)}")

    downloaded_count = 0
    failed_files = []

    for file in tqdm(drive_list, desc="Download"):
        try:
            file_obj = drive.CreateFile({'id': file['id']})
            fname = file["title"]
            outName = os.path.join(dl_folder, fname)
            file_obj.GetContentFile(outName)
            file_obj.Delete()
            downloaded_count += 1
        except Exception as e:
            print(f"\n✗ Fehler beim Download von {fname}: {str(e)}")
            failed_files.append(fname)

    # Lösche leeren GEE-Ordner
    try:
        drive.CreateFile({'id': gee_folder_id}).Delete()
        print(f"\n✓ Ordner '{export_folder_name}' aus Google Drive gelöscht")
    except Exception as e:
        print(f"\n⚠️  Ordner konnte nicht gelöscht werden: {str(e)}")

    print("\n" + "=" * 70)
    print("✓ DOWNLOAD ABGESCHLOSSEN")
    print("=" * 70)
    print(f"  Heruntergeladene Dateien: {downloaded_count}")
    if failed_files:
        print(f"  Fehlgeschlagen: {failed_files}")
    print(f"  Zielordner: {dl_folder}")

    for fname in sorted(os.listdir(dl_folder)):
        fpath = os.path.join(dl_folder, fname)
        size_mb = os.path.getsize(fpath) / (1024**2)
        print(f"  ✓ {fname} ({size_mb:.2f} MB)")

Suche nach MODIS_GEE_* Export-Ordnern auf Google Drive...
✓ 2 MODIS_GEE_* Ordner gefunden:
  1. MODIS_GEE_20260310_140043 (ID: 1om8Ph0YseapxXe__qMNv1BkA6lpTd8PJ)
  2. MODIS_GEE_20260219_173646 (ID: 1MGtNJTXtQasXtNvs294D4uPAk7Y9gjuS)

✓ Neuester Ordner: MODIS_GEE_20260310_140043
Anzahl Dateien im Ordner: 1


Download: 100%|██████████| 1/1 [00:06<00:00,  6.52s/it]



✓ Ordner 'MODIS_GEE_20260310_140043' aus Google Drive gelöscht

✓ DOWNLOAD ABGESCHLOSSEN
  Heruntergeladene Dateien: 1
  Zielordner: \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\04_Landcover\MODIS_GEE_tiles
  ✓ MODIS_IGBP_2001_2024_500m_wildE.tif (83.83 MB)


In [ ]:
# === Validierung gegen Grid-Referenz (woody_cover_500m_3035.tif) ===

woody_path = os.path.join(workDir, r"_Runs\02_Woody_Cover\woody_cover_500m_3035.tif")
modis_path = os.path.join(workDir, r"_Runs\04_Landcover\MODIS_GEE_tiles\MODIS_IGBP_2001_2024_500m_wildE.tif")

print("=" * 70)
print("VALIDIERUNG")
print("=" * 70)

with rasterio.open(woody_path) as ref:
    ref_shape = (ref.height, ref.width)
    ref_crs   = ref.crs
    ref_res   = ref.res

with rasterio.open(modis_path) as modis:
    modis_shape = (modis.height, modis.width)
    modis_crs   = modis.crs
    modis_res   = modis.res
    modis_bands = modis.count

print(f"  Grid-Referenz  : {ref_shape}, CRS={ref_crs}, res={ref_res}")
print(f"  MODIS IGBP     : {modis_shape}, CRS={modis_crs}, res={modis_res}, Bänder={modis_bands}")
print()

ok_shape = ref_shape == modis_shape
ok_crs   = ref_crs   == modis_crs
ok_bands = modis_bands == 24

print(f"  Shape Match   : {'✓' if ok_shape else '✗ FEHLER — Resampling nötig?'}")
print(f"  CRS Match     : {'✓' if ok_crs   else '✗ FEHLER'}")
print(f"  Bänder (= 24) : {'✓' if ok_bands else f'✗ FEHLER ({modis_bands} statt 24)'}")

if ok_shape and ok_crs and ok_bands:
    print()
    print("=" * 70)
    print("✓✓✓ PREPROCESSING STEP 4 ABGESCHLOSSEN ✓✓✓")
    print("=" * 70)
    print(f"  Output: {modis_path}")
    print("  → Weiter mit: 05_analysis_eco_lc.ipynb")
    print("=" * 70)
else:
    raise RuntimeError("Validierung fehlgeschlagen — MODIS IGBP passt nicht auf Grid-Referenz!")


VALIDIERUNG
  Grid-Referenz  : (9660, 10483), CRS=EPSG:3035, res=(500.0, 500.0)
  MODIS IGBP     : (9660, 10483), CRS=EPSG:3035, res=(500.0, 500.0), Bänder=24

  Shape Match   : ✓
  CRS Match     : ✓
  Bänder (= 24) : ✓

✓✓✓ PREPROCESSING STEP 4 ABGESCHLOSSEN ✓✓✓
  Output: \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\04_Landcover\MODIS_GEE_tiles\MODIS_IGBP_2001_2024_500m_wildE.tif
  → Weiter mit: 05_analysis_eco_lc.ipynb
